# NZZ ContextAI — RAG Pipeline Test

In [1]:
import sys
sys.path.insert(0, "../src")

from config import (
    CHROMA_PATH, CHROMA_COLLECTION,
    EMBEDDING_MODEL, RERANKER_MODEL, USE_RERANKING,
    EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)
from embed import get_chroma_collection
from retrieval import load_models, retrieve
from generate import generate

print("Imports OK")

Imports OK


## 1 — Modelle & Datenbank laden

In [2]:
collection = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
model, reranker = load_models(use_reranking=USE_RERANKING)

print(f"Chunks in ChromaDB: {collection.count():,}")
print(f"Embedding-Modell:   {EMBEDDING_MODEL}")
print(f"Reranker:           {RERANKER_MODEL if USE_RERANKING else 'deaktiviert'}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks in ChromaDB: 3,140
Embedding-Modell:   Qwen/Qwen3-Embedding-0.6B
Reranker:           cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## 2 — Einzelne Anfrage testen

In [3]:
QUERY = "Wie werden ukrainische Kinder in Russland umerzogen?"

In [4]:
chunks = retrieve(
    QUERY, model, collection, reranker,
    top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
    top_k_rerank=EVAL_TOP_K_FINAL,
)

print(f"Top-{len(chunks)} Quellen fuer: '{QUERY}'\n")
for i, c in enumerate(chunks, 1):
    score = c.get("rerank_score", c.get("similarity_score", 0))
    print(f"  [{i}] {c['title'][:65]}")
    print(f"       Artikel-ID: {c['article_id']}  |  Datum: {c.get('published_date', '-')}  |  Score: {score:.3f}")

Top-5 Quellen fuer: 'Wie werden ukrainische Kinder in Russland umerzogen?'

  [1] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 8.188
  [2] Ukrainische Kinder in Russland aufgespürt: Wie KI bei der Suche h
       Artikel-ID: 336520876  |  Datum: 2025-12-13  |  Score: 6.422
  [3] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 5.426
  [4] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 5.176
  [5] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 4.416


## 3 — Antwort generieren

In [5]:
answer = generate(QUERY, chunks)
print(answer)

**Direkte Antwort**
Russland umerzieht ukrainische Kinder zu patriotischen Russen durch eine gezielte Umerziehung. Diese Kinder werden oft aus ihrer Familie entfernt und in ein Waisenhaus gebracht, wo sie russische Werte und Identität erlernten. Die Kinder werden auch dazu gezwungen, ihre ukrainische Sprache und Kultur aufzugeben und sich als Russen zu identifizieren.

**Kontextualisierung**
Die Umerziehung von ukrainischen Kindern in Russland ist ein Teil des russischen Kriegsmanövers gegen die Ukraine. Die Kinder werden oft aus ihrer Familie entfernt, weil ihre Eltern oder Verwandten als Feinde des russischen Regimes angesehen werden. Die Kinder werden dann in ein Waisenhaus gebracht, wo sie russische Werte und Identität erlernten. Dieser Prozess kann auch dazu führen, dass die Kinder ihre ukrainische Sprache und Kultur aufgeben und sich als Russen identifizieren.

**Quellenübersicht**
Die Quellen für diese Informationen sind:

* [1] Artikel-ID: 336520888 | Titel: Die stummen Opfer d

## 4 — Mehrere Ground-Truth-Queries durchlaufen

In [6]:
import json
from config import EVAL_GROUND_TRUTH

with open(EVAL_GROUND_TRUTH) as f:
    ground_truth = [json.loads(line) for line in f if line.strip()]

print(f"{len(ground_truth)} Queries geladen")

14 Queries geladen


In [7]:
N = 3

for sample in ground_truth[:N]:
    query = sample["query"]
    relevant = set(sample["relevant_article_ids"])

    results = retrieve(
        query, model, collection, reranker,
        top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
        top_k_rerank=EVAL_TOP_K_FINAL,
    )

    top1_id = results[0]["article_id"] if results else ""
    hit = "HIT" if top1_id in relevant else "MISS"

    print(f"[{hit}] {query}")
    print(f"  Erwartet: {relevant}")
    print(f"  Top-1:    {top1_id} — {results[0]['title'][:60] if results else '-'}")

    answer = generate(query, results)
    print(f"  Antwort:  {answer}")
    print()

[HIT] Was erwarten Schweizer Anlageprofis für die Börsenmärkte im Jahr 2026?
  Erwartet: {'336541061'}
  Top-1:    336541061 — «Wir werden in spätestens 18 Monaten wieder Negativzinsen ha
  Antwort:  **Direkte Antwort**
Die Schweizer Wirtschaft erwarten für 2026 ein reales Wachstum des Bruttoinlandprodukts (BIP) von 1,1 Prozent, wie sie am Montag mitteilten. Die Risiken bleiben gross.

**Kontextualisierung**
Die Zollvereinbarung mit den USA hat die Aussichten für die Schweizer Wirtschaft etwas aufgehellt. Sowohl das Staatssekretariat für Wirtschaft (Seco) wie auch das KOF-Institut der ETH Zürich erwarten für 2026 ein reales Wachstum des BIP von 1,1 Prozent.

**Quellenübersicht**
- [1] Artikel-ID: 336541061 | Titel: «Wir werden in spätestens 18 Monaten wieder Negativzinsen haben»: Anlageprofis sehen mit gemischten Gefühlen ins Jahr 2026
- [2] Artikel-ID: 336525296 | Titel: Der Zoll-Deal mit den USA hellt die Aussichten für die Schweizer Wirtschaft auf – trotzdem dürfte 2026 ein schwache